# 30 · Evaluate all compatible HPO final runs
Evaluates only matching track/protocol runs and reports baseline/tuned mean and standard deviation across seeds.

In [ ]:
DATASET_TRACK = "2class"
EVALUATOR_VERSION = "v2"
BENCHMARK_TRACK = "controlled"
EVALUATE_MISSING = False

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
IN_KAGGLE = not IN_COLAB and bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or Path("/kaggle/working").is_dir()
)
NOTEBOOK_PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

repository_default = (
    Path("/content/aerial-object-detection-benchmark")
    if IN_COLAB
    else Path("/kaggle/working/aerial-object-detection-benchmark")
    if IN_KAGGLE
    else Path.cwd()
)
repository_override = os.environ.get("BENCHMARK_REPO_ROOT")
repository_candidates = (
    [Path(repository_override).expanduser()]
    if repository_override
    else [Path.cwd(), *Path.cwd().parents, repository_default]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in repository_candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "__init__.py").is_file()
    ),
    repository_default.resolve(),
)
git_probe = subprocess.run(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--is-inside-work-tree"],
    check=False, capture_output=True, text=True,
)
if git_probe.returncode != 0:
    if NOTEBOOK_PLATFORM == "local":
        raise RuntimeError(
            "Run this notebook from the repository or set BENCHMARK_REPO_ROOT."
        )
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)],
        check=True,
    )
else:
    current_commit = subprocess.check_output(
        ["git", "-C", str(REPO_PATH), "rev-parse", "HEAD"], text=True
    ).strip()
    print(f"Using selected repository commit {current_commit}.")
sys.path.insert(0, str(REPO_PATH))

from src.notebook_environment import setup_notebook_environment
notebook_environment = setup_notebook_environment(
    REPO_PATH,
    platform=NOTEBOOK_PLATFORM,
    use_google_drive=True,
    requirements_file=None,
    smoke_test=SMOKE_TEST,
)
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
print(notebook_environment.as_dict())
from src.config.benchmark_tracks import load_track_config
from src.subprocess_utils import run_checked
track_config = load_track_config(REPO_PATH, BENCHMARK_TRACK)
from src.evaluation.versioned import evaluate_prediction_artifact
print(f"Evaluator contract: {EVALUATOR_VERSION}")
if BENCHMARK_TRACK != "controlled":
    raise RuntimeError("Legacy evaluator accepts controlled artifacts only")
if SMOKE_TEST:
    result = {"status": "guarded", "dataset_track": DATASET_TRACK}
else:
    command = [sys.executable, "-m", "scripts.evaluate_all_models", "--drive-root", str(DRIVE_ROOT), "--dataset-track", DATASET_TRACK]
    if EVALUATE_MISSING:
        command.append("--evaluate-missing")
    run_checked(command, cwd=REPO_PATH, stage="evaluate_all_models", python_executable=sys.executable)
    result = {"status": "complete", "report": str(DRIVE_ROOT / "reports" / "comparison" / "two_stage_random_hpo_v1" / DATASET_TRACK / "comparison.json")}
result